In [61]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV

import warnings
warnings.filterwarnings('ignore')

In [36]:
df = pd.read_parquet(r'D:\zepto_ETA_prediction\data\features\final_features.parquet', engine="pyarrow")

In [37]:
df.head()

,distance_km,rider_batch_count,traffic,weather,traffic_batch,is_peak,order_hour,distance_bucket,weight_class,delivery_time_min
0,7.64,1,high,clear,3,0,8,long,light\r,20.49
1,7.42,1,medium,clear,2,1,13,long,medium\r,16.76
2,2.79,1,high,clear,3,1,19,medium,medium\r,11.30
3,3.40,1,medium,clear,2,1,18,medium,heavy\r,14.28
4,4.58,1,low,clear,1,0,14,medium,heavy\r,16.27


In [38]:
le = LabelEncoder()

df['traffic'] = le.fit_transform(df['traffic'])
df['weather'] = le.fit_transform(df['weather'])
df['distance_bucket'] = le.fit_transform(df['distance_bucket'])
df['weight_class'] = le.fit_transform(df['weight_class'])

In [39]:
df.head()

,distance_km,rider_batch_count,traffic,weather,traffic_batch,is_peak,order_hour,distance_bucket,weight_class,delivery_time_min
0,7.64,1,0,0,3,0,8,0,1,20.49
1,7.42,1,2,0,2,1,13,0,2,16.76
2,2.79,1,0,0,3,1,19,1,2,11.30
3,3.40,1,2,0,2,1,18,1,0,14.28
4,4.58,1,1,0,1,0,14,1,0,16.27


## Train test split

In [40]:
X= df.drop('delivery_time_min', axis=1)
y=df.delivery_time_min

In [41]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## mean prdiction

In [42]:
y_pred_baseline = [y_train.mean()] * len(y_test)

In [43]:
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

In [44]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("RF MAE:", mae_rf)
print("RF RMSE:", rmse_rf)

RF MAE: 0.8372047769352202
RF RMSE: 1.2116289655621428


In [45]:
## XGBoost

In [46]:
xgb = XGBRegressor(random_state=42)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

In [47]:
lgbm = LGBMRegressor(random_state=42)
lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_test)
mae_lgbm = mean_absolute_error(y_test, y_pred_lgbm)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004023 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 304
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 9
[LightGBM] [Info] Start training from score 13.346627


In [48]:
print("Baseline MAE:", mean_absolute_error(y_test, y_pred_baseline))
print("RF MAE:", mae_rf)
print("XGB MAE:", mae_xgb)
print("LGBM MAE:", mae_lgbm)

Baseline MAE: 4.1842114451
RF MAE: 0.8372047769352202
XGB MAE: 0.7533240010404586
LGBM MAE: 0.7529431288044974


In [49]:
df_test = X_test.copy()
df_test['actual'] = y_test
df_test['pred'] = y_pred_lgbm

df_test['error'] = abs(df_test['actual'] - df_test['pred'])

In [50]:
df_test.sort_values(by='error', ascending=False).head(10)

,distance_km,rider_batch_count,traffic,weather,traffic_batch,is_peak,order_hour,distance_bucket,weight_class,actual,pred,error
10064,4.12,2,0,0,6,1,19,1,2,10.04,16.757581,6.717581
2662,5.27,1,0,0,3,0,9,0,0,26.49,20.680925,5.809075
8994,7.74,1,0,0,3,0,8,0,2,15.95,21.085961,5.135961
80944,4.23,1,0,0,3,0,20,1,2,18.99,13.914880,5.075120
63691,4.34,1,0,2,3,1,19,1,1,15.56,20.576530,5.016530
90256,6.86,1,0,0,3,0,8,0,1,14.66,19.659985,4.999985
82634,3.20,2,0,0,6,1,19,1,2,19.79,14.841654,4.948346
90411,3.10,2,0,0,6,1,19,1,0,14.82,19.664934,4.844934
60625,4.62,2,0,0,6,0,20,1,1,13.10,17.929755,4.829755
78071,7.36,1,0,0,3,0,8,0,1,25.29,20.461081,4.828919


In [51]:
import pandas as pd

importance = pd.Series(rf.feature_importances_, index=X.columns)
importance.sort_values(ascending=False).head(10)

distance_km          0.644337
traffic_batch        0.110904
weight_class         0.085127
weather              0.085026
rider_batch_count    0.052657
order_hour           0.008980
distance_bucket      0.007165
traffic              0.003495
is_peak              0.002307
dtype: float64

In [52]:
## Tuning

In [74]:
useful_features = [
    'distance_km',
    'rider_batch_count',
    'traffic_batch',
    'weather',
    'weight_class',
    'delivery_time_min',
    'distance_per_batch'
]

In [75]:
df = df[useful_features]

In [76]:
X = df.drop('delivery_time_min', axis=1)
y = df.delivery_time_min

In [77]:
X_train, X_test, y_train, y_test =train_test_split(X, y, test_size=0.2, random_state=42)

In [78]:
## HYPERPARAMETER TUNING

In [79]:
param_grid = {
    'n_estimators': [200, 300],
    'learning_rate': [0.03, 0.05],
    'max_depth': [4, 5],
    'num_leaves': [20, 31],
    'min_child_samples': [20, 30],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

In [80]:
model = LGBMRegressor(random_state=42)

grid = GridSearchCV(
    model,
    param_grid,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001636 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 527
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 6
[LightGBM] [Info] Start training from score 13.346627
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

In [71]:
df['distance_per_batch'] = df['distance_km'] / (df['rider_batch_count'] + 1)

In [81]:
mae = mean_absolute_error(y_test, y_pred)
print("Tuned MAE:", mae)

Tuned MAE: 0.7469722970599236
